# 1. Conversión de Dataset IMVIA a Formato YOLO

Este notebook procesa los videos del dataset IMVIA, extrae los frames y genera las etiquetas YOLO basadas en los archivos de anotación originales.

**Entorno Recomendado:** `tfmvc`

In [ ]:
import cv2
import os
import shutil
from pathlib import Path
from tqdm import tqdm
import random

# Rutas
SOURCE_BASE = Path("/Volumes/Echo 13 SSD/TFM/images/CAIDAS/FALLDATASET_IMVIA")
TARGET_BASE = Path("/Volumes/Echo 13 SSD/YOLO_DATASET_UNIFIED")

# Clases: 0=Fall, 1=No Fall
CLASS_FALL = 0
CLASS_NORMAL = 1

def convert_imvia():
    subdirs = ["Coffee_room_01", "Coffee_room_02", "Home_01", "Home_02", "Lecture_room", "Office"]
    all_tasks = []
    
    for sd in subdirs:
        video_dir = SOURCE_BASE / sd / sd / "Videos"
        if not video_dir.exists(): video_dir = SOURCE_BASE / sd / "Videos"
        if not video_dir.exists(): continue
            
        annot_dir = video_dir.parent / "Annotation_files"
        if not annot_dir.exists(): annot_dir = video_dir.parent / "Annotations_files"
        if not annot_dir.exists(): continue

        for v_file in video_dir.glob("*.avi"):
            v_name = v_file.stem
            a_file = annot_dir / f"{v_name}.txt"
            if a_file.exists():
                all_tasks.append((v_file, a_file, sd))

    print(f"Total de videos a procesar: {len(all_tasks)}")
    
    for v_path, a_path, sd_name in tqdm(all_tasks, desc="Procesando videos"):
        with open(a_path, 'r', encoding='latin-1') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        
        if len(lines) < 3: continue
        try:
            fall_start = int(lines[0])
            fall_end = int(lines[1])
        except ValueError: continue
            
        frame_annots = {}
        for l in lines[2:]:
            parts = l.split(',')
            if len(parts) >= 6:
                f_idx = int(parts[0])
                coords = [int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])]
                frame_annots[f_idx] = coords

        cap = cv2.VideoCapture(str(v_path))
        if not cap.isOpened(): continue
        
        width  = cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 320
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 240
        
        f_count = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            
            f_count += 1
            if f_count in frame_annots:
                coords = frame_annots[f_count]
                x1, y1, x2, y2 = coords
                if x1 == 0 and y1 == 0 and x2 == 0 and y2 == 0: continue
                
                cls = CLASS_FALL if (fall_start <= f_count <= fall_end) else CLASS_NORMAL
                
                bw = abs(x2 - x1)
                bh = abs(y2 - y1)
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                
                y_cx, y_cy = cx / width, cy / height
                y_bw, y_bh = bw / width, bh / height
                
                base_name = f"IMVIA_{sd_name}_{v_path.stem}_f{f_count:04d}"
                split = "train" if random.random() < 0.8 else "val"
                
                img_dir = TARGET_BASE / "images" / split
                img_dir.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(img_dir / f"{base_name}.jpg"), frame)
                
                lbl_dir = TARGET_BASE / "labels" / split
                lbl_dir.mkdir(parents=True, exist_ok=True)
                with open(lbl_dir / f"{base_name}.txt", 'w') as lf:
                    lf.write(f"{cls} {y_cx:.6f} {y_cy:.6f} {y_bw:.6f} {y_bh:.6f}\n")
                    
        cap.release()

if __name__ == "__main__":
    convert_imvia()
    print("Conversión finalizada.")